In [0]:
lst= dbutils.fs.ls("/Volumes/workspace/olist/volume_olist")
display(lst)

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist/volume_olist/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1774936917000
dbfs:/Volumes/workspace/olist/volume_olist/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1774936926000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1774936916000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1774936914000
dbfs:/Volumes/workspace/olist/volume_olist/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1774936913000
dbfs:/Volumes/workspace/olist/volume_olist/product_category_name_translation.csv,product_category_name_translation.csv,2613,1774936913000


#### Olist_Customers

In [0]:
%sql 
SELECT *
FROM workspace.bronze.olist_customers
LIMIT 10

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


##### Type Casting

In [0]:
%sql 
SELECT 
      *
FROM workspace.bronze.olist_customers
LIMIT 5

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [0]:

%sql
DESCRIBE workspace.bronze.olist_customers;

col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_zip_code_prefix,int,null
customer_city,string,null
customer_state,string,null


Only 1 field, customer_zip_code_prefix needs to be casted,despite it only consists of integer values, 
there are no need for arithmetic operations for zip code values, therefore it is better be string value

In [0]:
%sql
SELECT 
    customer_id
FROM workspace.bronze.olist_customers
WHERE customer_id != trim(customer_id)

customer_id


##### String Cleaning
- Unwanted Spaces
- Case correction

All columns are string, all fields will trimmed from white spaces and will be lower case 

In [0]:
%sql
SELECT 
      *
FROM workspace.bronze.olist_customers
LIMIT 5

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


##### NULL handling 

In [0]:
%sql 
SELECT 
    COUNT(*)
FROM workspace.bronze.olist_customers
WHERE customer_id IS NULL

COUNT(*)
0


In [0]:
%sql 
SELECT 
    COUNT(*) - COUNT(customer_id),
    COUNT(*) - COUNT(customer_unique_id),
    COUNT(*) - COUNT(customer_zip_code_prefix),
    COUNT(*) - COUNT(customer_city),
    COUNT(*) - COUNT(customer_state)
FROM workspace.bronze.olist_customers

COUNT(*)-COUNT(customer_id),COUNT(*)-COUNT(customer_unique_id),COUNT(*)-COUNT(customer_zip_code_prefix),COUNT(*)-COUNT(customer_city),COUNT(*)-COUNT(customer_state)
0,0,0,0,0


There is no null value 

##### Deduplication 

In [0]:
%sql -- Find duplicates 
SELECT
  COUNT(*) - COUNT(DISTINCT customer_id) as duplicate_customer_id,
  COUNT(*) - COUNT(DISTINCT customer_unique_id) as customer_unique_id,
  COUNT(*) - COUNT(DISTINCT customer_zip_code_prefix) as customer_zip_code_prefix,
  COUNT(*) - COUNT(DISTINCT customer_city) as customer_city ,
  COUNT(*) - COUNT(DISTINCT customer_state)  as customer_state 
FROM workspace.bronze.olist_customers


duplicate_customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,3345,84447,95322,99414


##### Date Standardization

No date 

customer_state,COUNT(*)
SP,41746
SC,3637
MG,11635
PR,5045
RJ,12852


##### Value Standardization

In [0]:
%sql
SELECT 
  customer_state,
  COUNT(*)
FROM workspace.bronze.olist_customers
GROUP BY customer_state
LIMIT 5

##### Column filtering

##### Row filtering 

#### Geo Loc
Actions:
   - Type Casting 
      - lat,lng converted into Float Type from String Type  
   - String Cleaning
      - Geolocation_city will be trimmed
   
   - NULL 
      - No NULL value observed in Columns 
   - Dedup 
      - Some rows are are duplicate entirely 
   - Date Standardication 
      - No date Value 
   - Value Standardization
     - 
      - No need , all values are numeric and 5 digits 

   


| Column                      | Type Casting  | Str. Cleaning | NULL       | Dedup| 
|---------------------------- |:-------------:| --------------| ---------- |------|
| geolocation_zip_code_prefix |    No         | No            |  No        | Yes  |
| geolocation_lng             |String to Flt. | Not applicable|  No        | Yes. |
| geolocation_lat             |String to Flt. | Not applicable|  No        | Yes. | 
| geolocation_city            |    No         |  Yes          |  No        | Yes  | 
| geolocation_staet           |    No         |  No           |  No        | Yes  | 


| Column                      | Value Standardization  | Str. Cleaning | NULL       | Dedup| 
|---------------------------- |-------------           | --------------| ---------- |------|
| geolocation_zip_code_prefix |    No                  | No            |  No        | Yes  |
| geolocation_lng             |String to Flt.          | Not applicable|  No        | Yes. |
| geolocation_lat             |String to Flt.          | Not applicable|  No        | Yes. | 
| geolocation_city            |    No                  |  Yes          |  No        | Yes  | 
| geolocation_staet           |    No                  |  No           |  No        | Yes  | 

##### Type Analysis 

In [0]:
%sql 
SELECT 
      * 
FROM 
    workspace.bronze.olist_geo_loc
LIMIT 5

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


In [0]:
%sql 
DESCRIBE workspace.bronze.olist_geo_loc

col_name,data_type,comment
geolocation_zip_code_prefix,string,null
geolocation_lat,string,null
geolocation_lng,string,null
geolocation_city,string,null
geolocation_state,string,null


In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_geo_loc
LIMIT 5

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


In [0]:
%sql 
SELECT 
    


##### String Cleaning
- geolocation_zip_code_prefix
- geolocation_city
    - has 1 record with unwanted spaces 
- geolocation_state 

In [0]:
%sql 
-- Check if geolocation_zip_code_prefix is has unwanted spaces
-- Result: No record has unwanted spaces 
SELECT 
  geolocation_zip_code_prefix
FROM workspace.bronze.olist_geo_loc
WHERE geolocation_zip_code_prefix != trim(geolocation_zip_code_prefix)

geolocation_zip_code_prefix


In [0]:
%sql 
-- Check if geolocation_city has unwanted spaces 
-- Result: One record has unwanted spaces in geolocation_city
SELECT 
  geolocation_city,
  length(geolocation_city) as original_length , 
  length(trim(geolocation_city)) as trimmed_length 
FROM workspace.bronze.olist_geo_loc
WHERE geolocation_city != trim(geolocation_city)

geolocation_city,original_length,trimmed_length
salvador,9,8


In [0]:
%sql 
-- Check if GeoLocation_state has unwanted spaces
-- Result: No unwanted spaces
SELECT
  geolocation_state,
  length(geolocation_state) as original_length , 
  length(trim(geolocation_state)) as trimmed_lengthgeo
FROM workspace.bronze.olist_geo_loc
WHERE geolocation_state != trim(geolocation_state)


geolocation_state,original_length,trimmed_lengthgeo


##### Null Handling

In [0]:
%sql 
-- Check how many Null values each column have 
-- No column have NULL value
SELECT
  COUNT(*) - COUNT(geolocation_zip_code_prefix) as zip_code_null,
  COUNT(*) - COUNT(geolocation_city) as city_null,
  COUNT(*) - COUNT(geolocation_state) as state_null,
  COUNT(*) - COUNT(geolocation_lat) as lat_null,
  COUNT(*) - COUNT(geolocation_lng) as lng_null

FROM workspace.bronze.olist_geo_loc

zip_code_null,city_null,state_null,lat_null,lng_null
0,0,0,0,0


##### Deduplication 

- Rows have duplicate values 


In [0]:
%sql 
SELECT 
  geolocation_zip_code_prefix,
  geolocation_city,
  geolocation_state,
  geolocation_lat,
  geolocation_lng, 
  COUNT(*)
FROM workspace.bronze.olist_geo_loc
GROUP BY geolocation_zip_code_prefix, geolocation_city, geolocation_state, geolocation_lat, geolocation_lng
HAVING COUNT(*) > 1

geolocation_zip_code_prefix,geolocation_city,geolocation_state,geolocation_lat,geolocation_lng,COUNT(*)
01037,sao paulo,SP,-23.54562128115268,-46.63929204800168,2
01046,sao paulo,SP,-23.546081127035535,-46.64482029837157,35
01046,sao paulo,SP,-23.54612896641469,-46.64295148361138,6
01047,sao paulo,SP,-23.546273112412678,-46.64122516971552,9
01013,sao paulo,SP,-23.546923208436723,-46.6342636964915,3
01029,sao paulo,SP,-23.543769055769133,-46.63427784085132,3
01011,sao paulo,SP,-23.547639550320632,-46.63603162315495,6
01013,sao paulo,SP,-23.547325128224376,-46.63418378613892,3
01012,sao paulo,SP,-23.548945985189434,-46.63467113292871,2
01039,sao paulo,SP,-23.541883009983316,-46.63991946670314,3


##### Value Standardization 
- Geolocation_zip_code_prefix
    - 5 digit ( put zero in front ) - True 
    - Numeric - True 

- Geolocation Latitude 
    - Precision 2 place 
    - range -90 and 90 
- Geolocation Longitude 
    - Precision 2 place 
    - range -180 and 180 

- Geolocation city 
   - Title case

- Geolocation_state 


In [0]:
%sql 
SELECT * 
FROM workspace.bronze.olist_geo_loc
LIMIT 5

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


In [0]:
%sql 
-- Check if geolocation_zip_code_prefix is shorter than 5 and all numeric
-- Result: All geolocation_zip_code_prefix are 5 digits and numeric 
SELECT 
    geolocation_zip_code_prefix,
    LENGTH(geolocation_zip_code_prefix), 
    try_cast(geolocation_zip_code_prefix as int) as int_zip 
FROM workspace.bronze.olist_geo_loc
WHERE LENGTH(geolocation_zip_code_prefix) < 5 OR try_cast(geolocation_zip_code_prefix as int) is NULL 

geolocation_zip_code_prefix,LENGTH(geolocation_zip_code_prefix),int_zip


In [0]:
%sql 
-- Check If geolocation_lat is between range -90 and 90
-- All values are between -90 and 90 
SELECT 
  geolocation_lat
FROM workspace.bronze.olist_geo_loc
WHERE try_cast(geolocation_lat as int )  not BETWEEN -90 and 90 

geolocation_lat
